# NeuroWorkflow: Inspecting the Network That Was Built

`NW_Analysis` answers *how did the cells fire*. `NW_NetworkView` answers a question
you have to ask first: **what did my connection rules actually produce?**

The two are not the same. A connection rule is a statement of intent; the SONATA
files are the result, and a random rule does not grant every neuron its average. This
notebook builds the familiar Brunel balanced network and finds that a sizeable share
of its excitatory cells receive **no inhibition at all** - which looks, from a raster
plot, exactly like a modelling mistake.

```
                      ┌──→ NW_Analysis      what the network DID
NW_SimConfig ─results─┤
                      └──→ NW_NetworkView   what the network IS
```

Both read the same `results`, so the viewer costs one extra edge and no extra run.

## 1. Build and run a Brunel balanced network

In [ ]:
from neuroworkflow import WorkflowBuilder
from neuroworkflow.nodes.network.NW_Population    import NW_Population
from neuroworkflow.nodes.network.NW_Connectivity  import NW_Connectivity
from neuroworkflow.nodes.simulation.NW_SimConfig  import NW_SimConfig
from neuroworkflow.nodes.analysis.NW_Analysis     import NW_Analysis
from neuroworkflow.nodes.analysis.NW_NetworkView  import NW_NetworkView

In [ ]:
exc  = NW_Population("PopExc")
inh  = NW_Population("PopInh")
conn = NW_Connectivity("Connectivity")
cfg  = NW_SimConfig("SimConfig")
ana  = NW_Analysis("Analysis")
view = NW_NetworkView("NetworkView")

neuron = dict(
    model_type     = "point_neuron",
    model_template = "nest:iaf_psc_alpha",
    location       = "VISp",
    layer          = "L4",
    nest_params = {
        "C_m": 250.0, "tau_m": 10.0, "t_ref": 2.0,
        "V_th": -55.0, "V_reset": -70.0, "E_L": -70.0,
        "I_e": 500.0,      # stands in for the external drive
    },
)
exc.configure(pop_name="exc", N=80, ei_type="exc", **neuron)
inh.configure(pop_name="inh", N=20, ei_type="inh", **neuron)

# 10% sparse, repeatable: seeded by the pathway number and the pair of node ids.
sparse = lambda n: (
    "lambda src, tgt: 1 if np.random.default_rng("
    f"[{n}, src['node_id'], tgt['node_id']]).random() < 0.1 else 0")

w, g = 40.0, 5.0
conn.configure(
    model_template="static_synapse", delay=1.5, syn_weight=w,
    connections=[
        {"source": "exc", "target": "exc", "connection_rule": sparse(0)},
        {"source": "exc", "target": "inh", "connection_rule": sparse(1)},
        {"source": "inh", "target": "exc", "connection_rule": sparse(2), "syn_weight": -g * w},
        {"source": "inh", "target": "inh", "connection_rule": sparse(3), "syn_weight": -g * w},
    ],
)

cfg.configure(simulator="pointnet", config_file="config_sparse.json",
              tstop_ms=1000.0, dt_ms=0.1, reports={})
ana.configure(plot_raster=True, plot_traces=False, plot_rate=False, save_figures=True)
view.configure(plot_matrix=True, plot_in_degree=True, save_figures=True)

In [ ]:
wf = WorkflowBuilder("NW_NetworkView_Inspect")
for node in [exc, inh, conn, cfg, ana, view]:
    wf.add_node(node)

wf.connect("PopExc", "population", "Connectivity", "populations")
wf.connect("PopInh", "population", "Connectivity", "populations")
wf.connect("Connectivity", "network", "SimConfig", "populations")

# Analysis and NetworkView both read the same results — one run, two questions.
wf.connect("SimConfig", "results", "Analysis",    "results")
wf.connect("SimConfig", "results", "NetworkView", "results")

wf.context["results_path"] = "./results/networkview_sparse"
workflow = wf.build()
assert workflow.execute(), "workflow failed — check the printed errors"
print("done")

## 2. What the rules actually produced

The node prints a line per projection while it runs. The ports carry the same numbers
as data, which is what the next cell reads.

In [ ]:
projections = view._output_ports["projections"].value
unconnected = view._output_ports["unconnected"].value

print(f"{'projection':12} {'pairs':>6} {'density':>8} {'in-degree':>12} {'no input':>9}")
for key, d in sorted(projections.items()):
    print(f"{key:12} {d['pairs']:6} {d['density_percent']:7.1f}% "
          f"{d['in_degree_min']:4}-{d['in_degree_max']:<3} (avg {d['in_degree_mean']:.1f}) "
          f"{unconnected[key]:7}")

### Reading it

Every projection sits at about the 10% the rule asked for, and the averages look
right. The last column is the one that matters: **the number of target neurons that
received nothing at all through that projection.**

`inh->exc` is where it bites. With 20 inhibitory neurons and a 10% chance each, a
target's expected input is 2 - and a run of twenty coin flips at p=0.1 comes up empty
often enough that a tenth of the excitatory population ends up with no inhibition.
Those cells are not balanced at all. In a raster they are simply the fast ones, and
nothing tells you why.

In [ ]:
n_exc = view._output_ports["populations"].value["exc"]
missing = unconnected["inh->exc"]
print(f"  excitatory neurons: {n_exc}")
print(f"  receiving no inhibition: {missing}  ({100 * missing / n_exc:.0f}%)")
print(f"  expected inputs per neuron: {projections['inh->exc']['in_degree_mean']:.1f}")

## 3. A rule that leaves nobody out

If every neuron must receive the same number of inputs, do not ask for a density -
ask for a degree. Seeding the generator with the **target** alone makes every source
agree on which few were chosen, so the count is exact rather than average.

Each run writes to its own `results_path`: changing the connections changes the
network, and rebuilding one into a directory already used in this session fails on a
file the previous run still holds open.

In [ ]:
fixed_in_degree = (
    "lambda src, tgt: 1 if src['node_id'] in "
    "np.random.default_rng([2, tgt['node_id']]).choice(20, 2, replace=False) else 0")

conn.configure(connections=[
    {"source": "exc", "target": "exc", "connection_rule": sparse(0)},
    {"source": "exc", "target": "inh", "connection_rule": sparse(1)},
    # every excitatory neuron receives from exactly 2 of the 20 inhibitory cells
    {"source": "inh", "target": "exc", "connection_rule": fixed_in_degree, "syn_weight": -g * w},
    {"source": "inh", "target": "inh", "connection_rule": sparse(3), "syn_weight": -g * w},
])
cfg.configure(config_file="config_fixed.json")
for node in workflow.nodes.values():
    node._context["results_path"] = "./results/networkview_fixed"

assert workflow.execute(), "re-run failed"
fixed = view._output_ports["projections"].value["inh->exc"]
print(f"\n  inh->exc now: in-degree {fixed['in_degree_min']}-{fixed['in_degree_max']}, "
      f"{view._output_ports['unconnected'].value['inh->exc']} neuron(s) with no input")

Every excitatory neuron now has exactly two inhibitory inputs. Same average as
before, no gaps.

Compare the two `NW_Analysis` lines above. The mean rate barely moves - averages
hide this - but the **CV falls sharply**, from around 0.8 to around 0.2. The spread
in the first run was not the network being irregular; it was a tenth of the
population running without a brake while the rest were balanced. Two populations of
cells were being averaged into one number.

## 4. What else the viewer reports

**`edge_types` and `edge_types_share_wiring`** matter for multi-receptor models. A
pathway with an AMPA and an NMDA component is *two edge types over the same pairs* -
one synapse with two components. Give those two projections different connection
rules by mistake and you get two edge types over *different* pairs, which is a
different model that runs and looks plausible. The viewer states which you have; see
`NW_MultiReceptor_EI_Network.ipynb` for a network with four receptors.

**`populations`** is read from the SONATA files rather than from the `N` you typed, so
it reflects what was built.

### Why there is no 3D view

The Allen Institute's [VND](https://www.ks.uiuc.edu/Research/vnd/) renders SONATA
networks in 3D. It cannot help here yet: networks built by `NW_Population` carry no
coordinates - no `x`, `y` or `z` in the node files - so there is nothing to place in
space. Positions would have to be written at build time first.